### Package Requirement:
pip install langchian chromadb pypdf sentence-transformers
pip install -U langchain-community

In [1]:
# %pip install -qU langchain langchain-community
%pip install -qU langchain-openai

Note: you may need to restart the kernel to use updated packages.


deepseek api:sk-6942ce89f6cc4616b64cc57b91c7620e

In [2]:
import importlib
import site
importlib.reload(site)


<module 'site' from 'c:\\Users\\Wenduo Zhang\\miniconda3\\envs\\study\\lib\\site.py'>

In [3]:
import getpass
import os

if "AZURE_OPENAI_API_KEY" not in os.environ:
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass(
        " qhfhcZwRWTGDykfNXzUoINKnPxI8lFiQzBne4vJcbLzvxupagXFHJQQJ99ALACHYHv6XJ3w3AAAAACOGS5fO"
    )
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://sweet-m55d9k6j-eastus2.openai.azure.com/openai/deployments/yu-gpt-4o/chat/completions?api-version=2025-01-01-preview"

api key: qhfhcZwRWTGDykfNXzUoINKnPxI8lFiQzBne4vJcbLzvxupagXFHJQQJ99ALACHYHv6XJ3w3AAAAACOGS5fO
endpoint: https://sweet-m55d9k6j-eastus2.openai.azure.com/openai/deployments/yu-gpt-4o/chat/completions?api-version=2025-01-01-preview

In [3]:
# Imported package
import os
from pathlib import Path
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings


from langchain.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_openai import AzureChatOpenAI

In [4]:
pdf_folder = "Sample_PDF"

### Class

In [5]:
class EmbeddingClient:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        print(f"[EmbeddingClient] Loading embedding model: {model_name}")
        self.model = HuggingFaceEmbeddings(model_name=model_name)
    
    def get_model(self):
        return self.model

In [19]:
class LLM_Client:
    def __init__(self, api_key: str, model_name: str = "gpt-4", provider: str = "openai", **kwargs):
        from langchain.chat_models import ChatOpenAI
        from langchain_deepseek import ChatDeepSeek
        from langchain_openai import AzureChatOpenAI

        self.provider = provider.lower()
        self.model_name = model_name

        if self.provider == "openai":
            self.llm = ChatOpenAI(openai_api_key=api_key, model_name=model_name)
        
        elif self.provider == "deepseek":
            self.llm = ChatDeepSeek(
                api_key=api_key, 
                model=model_name,
                max_tokens=kwargs.get('max_tokens')
                )
        
        elif self.provider == "azure":
            self.llm = AzureChatOpenAI(
                api_key=api_key,
                azure_endpoint=kwargs.get("endpoint"),
                deployment_name=kwargs.get("deployment_name"),
                api_version=kwargs.get("api_version")
            )
        
        else:
            raise ValueError(f"Unsupported provider: {self.provider}")
        

    def call_llm_api(self, prompt):
        return self.llm.predict(prompt)

### Function

In [8]:
def find_all_pdfs(folder_path):
    pdf_files = list(Path(folder_path).rglob("*.pdf"))
    return pdf_files

In [9]:
def load_and_split_pdf(pdf_paths, chunk_size=500, chunk_overlap=50):
    all_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    print(f"Found {len(pdf_paths)} PDF files.")
    for path in pdf_paths:
        print(f"   -> processing: {path.name}")
        loader = PyPDFLoader(str(path))
        pages = loader.load()
        docs = splitter.split_documents(pages)
        # add source info to metadata
        for doc in docs:
            doc.metadata['source'] = path.name
        all_docs.extend(docs)
    
    print(f"Total chunks created: {len(all_docs)}")
    return all_docs


In [10]:
def embed_and_store(docs, embedding_model, persist_dir="./vector_db"):

    print("Storing all documents in Chroma vector DB...")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embedding_model,
        persist_directory=persist_dir
    )
    vectorstore.persist()
    print(f"Vector DB stored at {persist_dir}")
    return vectorstore

In [11]:
def query_vectorstore(vectorstore, query, k=3):
    print(f"\n Query: \"{query}\"")
    results = vectorstore.similarity_search(query, k=k)
    for i, doc in enumerate(results):
        print(f"\nResult {i+1}:\n{doc.page_content})")


In [12]:
def build_answer(llm_client, query, top_docs):
    context = "\n\n".join([doc.page_content for doc in top_docs])
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template="""
You are an expert assistant. Answer the question using the context below.

Context:
{context}

Question:
{question}
""")
    chain = LLMChain(llm=llm_client.llm, prompt=prompt_template)
    return chain.run(context=context, question=query)

### Test

In [19]:
# pdf_paths = find_all_pdfs(pdf_folder)
# docs = load_and_split_pdf(pdf_paths)
# vectorstore = embed_and_store(docs)
# query_vectorstore(vectorstore, "What is this document about?")

In [ ]:
API_KEY = "qhfhcZwRWTGDykfNXzUoINKnPxI8lFiQzBne4vJcbLzvxupagXFHJQQJ99ALACHYHv6XJ3w3AAAAACOGS5fO" 
END_POINT = "https://sweet-m55d9k6j-eastus2.openai.azure.com/openai/deployments/yu-gpt-4o/chat/completions?api-version=2025-01-01-preview"
DEPLOYMENT_NAME = "yu-gpt-4o"
API_VERSION = "2025-01-01-preview"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
CHUNK_SIZES = [800, 1000, 1500]
CHUNK_OVERLAP = 200
VECTOR_DB_DIR = "./vector_db"
PDF_PATH = "sample.pdf"
QUERIES = [
    "What is the main topic of this document?",
    "What are the key findings?",
    "How does the author support the argument?",
    "What are the limitations discussed?",
    "What is the conclusion?"
]

In [35]:
embed_client = EmbeddingClient(model_name=EMBEDDING_MODEL_NAME)
llm_client = LLM_Client(api_key=API_KEY, model_name=DEPLOYMENT_NAME, provider='azure', endpoint=END_POINT, deployment_name=DEPLOYMENT_NAME, api_version=API_VERSION)

pdf_paths = find_all_pdfs(pdf_folder)
for chuck_size in CHUNK_SIZES:
    print(f"Processing chunk size: {chuck_size}")
    chunks = load_and_split_pdf(pdf_paths=pdf_paths, chunk_size=chuck_size, chunk_overlap=CHUNK_OVERLAP)
    db =  embed_and_store(chunks, embed_client.get_model(), f"{VECTOR_DB_DIR}_{chuck_size}")
    for i, query in enumerate(QUERIES):
        top_docs = db.similarity_search(query, k=3)
        print(f"\nQuery {i+1}: {query}")
        print(" LLM Answer:")
        print(build_answer(llm_client, query, top_docs))

[EmbeddingClient] Loading embedding model: all-MiniLM-L6-v2
Processing chunk size: 800
Found 1 PDF files.
   -> processing: Sample.pdf
Total chunks created: 99
Storing all documents in Chroma vector DB...
Vector DB stored at ./vector_db_800

Query 1: What is the main topic of this document?
 LLM Answer:
The main topic of this document appears to be a collection of academic references spanning multiple disciplines. These include topics related to corporate governance and corporate social performance, copyright law in historical context, semantic reference in philosophy, human resource management in Taiwan, and the workings of open-source software, particularly user-to-user assistance. It serves as a bibliographic compilation or reference list for scholarly works across varied fields.

Query 2: What are the key findings?
 LLM Answer:
The key findings from the provided context are:

1. **Vagueness in Defining Disruptive Innovation**: The definition of "disruptive innovation" is unclear an

In [14]:
API_KEY = "sk-6942ce89f6cc4616b64cc57b91c7620e" 
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
CHUNK_SIZES = [800, 1000, 1500]
CHUNK_OVERLAP = 200
VECTOR_DB_DIR = "./vector_db"
PDF_PATH = "sample.pdf"
QUERIES = [
    "What is the main topic of this document?",
    "What are the key findings?",
    "How does the author support the argument?",
    "What are the limitations discussed?",
    "What is the conclusion?"
]

In [15]:
import getpass
import os

if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Enter your DeepSeek API key: ")

In [20]:
embed_client = EmbeddingClient(model_name=EMBEDDING_MODEL_NAME)
llm_client = LLM_Client(api_key=API_KEY, model_name= "deepseek-chat", provider= "deepseek")

pdf_paths = find_all_pdfs(pdf_folder)
for chuck_size in CHUNK_SIZES:
    print(f"Processing chunk size: {chuck_size}")
    chunks = load_and_split_pdf(pdf_paths=pdf_paths, chunk_size=chuck_size, chunk_overlap=CHUNK_OVERLAP)
    db =  embed_and_store(chunks, embed_client.get_model(), f"{VECTOR_DB_DIR}_{chuck_size}")
    for i, query in enumerate(QUERIES):
        top_docs = db.similarity_search(query, k=3)
        print(f"\nQuery {i+1}: {query}")
        print(" LLM Answer:")
        print(build_answer(llm_client, query, top_docs))

[EmbeddingClient] Loading embedding model: all-MiniLM-L6-v2
Processing chunk size: 800
Found 1 PDF files.
   -> processing: Sample.pdf
Total chunks created: 99
Storing all documents in Chroma vector DB...


C:\Users\Wenduo Zhang\AppData\Local\Temp\ipykernel_30636\7987057.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()
C:\Users\Wenduo Zhang\AppData\Local\Temp\ipykernel_30636\3921127897.py:14: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm_client.llm, prompt=prompt_template)
C:\Users\Wenduo Zhang\AppData\Local\Temp\ipykernel_30636\3921127897.py:15: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return chain.run(context=context, question=query)


Vector DB stored at ./vector_db_800

Query 1: What is the main topic of this document?
 LLM Answer:
The document appears to be a list of academic references or citations from various scholarly articles. The main topic is not explicitly stated, but the references cover a range of subjects, including:

1. **Human Resource Management** (e.g., "The case of Taiwan" in *International Journal of Human Resource Management*).  
2. **Corporate Governance and Social Performance** (e.g., Johnson & Greening, 1999, in *Academy of Management Journal*).  
3. **Copyright Law** (e.g., Joyce & Patterson, 2003, in *Emory Law Journal*).  
4. **Philosophy of Language** (e.g., Kripke, 1977, in *Midwest Studies in Philosophy*).  
5. **Open Source Software** (e.g., Lakhani & Von Hippel, 2003, in *Research Policy*).  

Since the document repeats the same set of references three times without additional context, it seems to be a bibliography or reference list rather than a cohesive article. The overarching theme